# SAM3 Canvas Mode Example

Canvas mode stitches a reference crop (with a bounding box annotation) alongside the target image into a single canvas.
The model segments objects in the target region that match the reference.

This notebook shows two variants:
1. **Canvas without text** — relies purely on the visual reference (`categories=["visual"]`)
2. **Canvas with text** — adds a real category name for better precision

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from instantlearn.data import Sample
from instantlearn.models.sam3 import SAM3
from instantlearn.models.sam3.sam3 import CanvasConfig, Sam3PromptMode
from instantlearn.visualizer import render_predictions

## 1. Define reference and targets

We annotate a single SUV in the reference image with an `[x1, y1, x2, y2]` bounding box.

In [ ]:
ASSETS = Path("assets/edsa_vehicles")
CATEGORY = "SUV"

# Reference: one SUV bounding box on frame 0077
# Roboflow annotations are (center_x, center_y, width, height) at 1920x1080.
# The image is 800x600, so we scale and convert to xyxy format.
ann_w, ann_h = 1920, 1080  # annotation resolution
img_w, img_h = 800, 600    # actual image resolution
sx, sy = img_w / ann_w, img_h / ann_h

# SUV annotation: center_x=1176.55, center_y=495.96, w=451.33, h=551.48
cx, cy, bw, bh = 1176.55, 495.96, 451.33, 551.48
x1 = (cx - bw / 2) * sx
y1 = (cy - bh / 2) * sy
x2 = (cx + bw / 2) * sx
y2 = (cy + bh / 2) * sy

ref_path = ASSETS / "EDSA-Orense-2_mp4-0077.jpg"
ref_bbox = np.array([[x1, y1, x2, y2]], dtype=np.float32)

# Target images
target_paths = [
    ASSETS / "EDSA-Orense-2_mp4-0151.jpg",
    ASSETS / "EDSA-Orense-2_mp4-0164.jpg",
]

print(f"Reference bbox (xyxy): {ref_bbox[0]}")

# Show reference with bbox
img = cv2.cvtColor(cv2.imread(str(ref_path)), cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(img)
bx1, by1, bx2, by2 = ref_bbox[0].astype(int)
ax.add_patch(plt.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=2, edgecolor="orange", facecolor="none"))
ax.set_title(f"Reference: {ref_path.name} — bbox for '{CATEGORY}'")
ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Canvas without text

Initialize SAM3 in Canvas mode. When `categories` is omitted, the model defaults to
visual-only matching — no text guidance.

In [ ]:
model = SAM3(
    prompt_mode=Sam3PromptMode.CANVAS,
    canvas_config=CanvasConfig(split_ratio=0.3, crop_padding=2.0),
    confidence_threshold=0.3,
    model_id="research21/sam3.1",
)

# Fit with visual-only reference — omit categories for no text guidance
ref_sample = Sample(image_path=str(ref_path), bboxes=ref_bbox)
model.fit(ref_sample)

# Predict on targets
targets = [Sample(image_path=str(p)) for p in target_paths]
preds_no_text = model.predict(targets)

for path, pred in zip(target_paths, preds_no_text):
    print(f"  {path.name}: {pred['pred_boxes'].shape[0]} detection(s)")

## 3. Canvas with text

Refit the same model with the real category name `"SUV"`. Text improves precision.

In [ ]:
# Refit with real category text
ref_with_text = Sample(
    image_path=str(ref_path),
    bboxes=ref_bbox,
    categories=[CATEGORY],
    category_ids=np.array([0]),
)
model.fit(ref_with_text)

preds_with_text = model.predict(targets)

for path, pred in zip(target_paths, preds_with_text):
    print(f"  {path.name}: {pred['pred_boxes'].shape[0]} detection(s)")

## 4. Compare results

In [ ]:
COLOR_MAP = {0: [30, 144, 255]}

fig, axes = plt.subplots(len(target_paths), 3, figsize=(20, 7 * len(target_paths)))

for row, path in enumerate(target_paths):
    img_rgb = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)

    # Original
    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f"Original — {path.name}", fontsize=11)
    axes[row, 0].axis("off")

    # Canvas no text
    vis = render_predictions(img_rgb, preds_no_text[row], COLOR_MAP, show_scores=True)
    n = preds_no_text[row]["pred_boxes"].shape[0]
    axes[row, 1].imshow(vis)
    axes[row, 1].set_title(f"Canvas Mode (No Label Information) — {n} det(s)", fontsize=11)
    axes[row, 1].axis("off")

    # Canvas with text
    vis = render_predictions(img_rgb, preds_with_text[row], COLOR_MAP, show_scores=True)
    n = preds_with_text[row]["pred_boxes"].shape[0]
    axes[row, 2].imshow(vis)
    axes[row, 2].set_title(f"Canvas Mode + Label Information '{CATEGORY}' — {n} det(s)", fontsize=11)
    axes[row, 2].axis("off")

fig.suptitle("SAM3 Canvas Mode: Visual-Only vs Visual + Text", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Notes

- **Canvas mode** works without any text category — it relies on the visual reference crop
- Providing a **real category name** improves precision (fewer false positives)
- Only a **single bounding box** on a **single reference image** is needed
- `CanvasConfig.split_ratio` controls how much of the canvas is allocated to the reference crop
- `CanvasConfig.crop_padding` controls the padding around the reference bbox when cropping

## 5. Multi-category: SUV + Motorcycle

Canvas mode supports multiple categories by passing a list of reference samples,
each with its own bounding box and category. The model detects both classes simultaneously.

In [ ]:
# Motorcycle annotation from the same frame: center_x=578.85, center_y=504.60, w=194.74, h=352.75
mcx, mcy, mbw, mbh = 578.85, 504.60, 194.74, 352.75
moto_bbox = np.array([[(mcx - mbw / 2) * sx, (mcy - mbh / 2) * sy,
                        (mcx + mbw / 2) * sx, (mcy + mbh / 2) * sy]], dtype=np.float32)

# Two reference samples — one per category.
# For multi-category, we must assign distinct category_ids even without text,
# otherwise both refs merge into a single category (all detections get the same label).
ref_suv = Sample(image_path=str(ref_path), bboxes=ref_bbox, category_ids=np.array([0]))
ref_moto = Sample(image_path=str(ref_path), bboxes=moto_bbox, category_ids=np.array([1]))

# Show both bboxes
img = cv2.cvtColor(cv2.imread(str(ref_path)), cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(img)
for bbox, color, label in [(ref_bbox[0], "orange", "SUV"), (moto_bbox[0], "cyan", "Motorcycle")]:
    bx1, by1, bx2, by2 = bbox.astype(int)
    ax.add_patch(plt.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=2, edgecolor=color, facecolor="none"))
    ax.text(bx1, by1 - 4, label, color=color, fontsize=10, fontweight="bold")
ax.set_title("Reference bboxes: SUV (orange) + Motorcycle (cyan)")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Multi-category without text
model.fit([ref_suv, ref_moto])
multi_no_text = model.predict(targets)

for path, pred in zip(target_paths, multi_no_text):
    print(f"  {path.name}: {pred['pred_boxes'].shape[0]} detection(s)")

# Multi-category with text
ref_suv_text = Sample(
    image_path=str(ref_path), bboxes=ref_bbox,
    categories=["SUV"], category_ids=np.array([0]),
)
ref_moto_text = Sample(
    image_path=str(ref_path), bboxes=moto_bbox,
    categories=["Motorcycle"], category_ids=np.array([1]),
)
model.fit([ref_suv_text, ref_moto_text])
multi_with_text = model.predict(targets)

for path, pred in zip(target_paths, multi_with_text):
    print(f"  {path.name}: {pred['pred_boxes'].shape[0]} detection(s)")

In [ ]:
MULTI_COLORS = {0: [255, 165, 0], 1: [0, 255, 255]}  # orange=SUV, cyan=Motorcycle

fig, axes = plt.subplots(len(target_paths), 3, figsize=(20, 7 * len(target_paths)))

for row, path in enumerate(target_paths):
    img_rgb = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)

    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f"Original — {path.name}", fontsize=11)
    axes[row, 0].axis("off")

    vis = render_predictions(img_rgb, multi_no_text[row], MULTI_COLORS, show_scores=True)
    n = multi_no_text[row]["pred_boxes"].shape[0]
    axes[row, 1].imshow(vis)
    axes[row, 1].set_title(f"Multi-category (no text) — {n} det(s)", fontsize=11)
    axes[row, 1].axis("off")

    vis = render_predictions(img_rgb, multi_with_text[row], MULTI_COLORS, show_scores=True)
    n = multi_with_text[row]["pred_boxes"].shape[0]
    axes[row, 2].imshow(vis)
    axes[row, 2].set_title(f"Multi-category + Label Info — {n} det(s)", fontsize=11)
    axes[row, 2].axis("off")

fig.suptitle("Multi-Category Canvas: SUV (orange) + Motorcycle (cyan)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()